# Part B, Q1: Dataset and Exploratory Data Analysis**Owner: Ying Xin (Chung Ying Xin)****Dataset.** Yelp Review Full, a balanced 30,000-review sample (6,000 reviews per starrating 1 to 5), remapped to 3-class sentiment: `negative` = 1-2 stars, `neutral` = 3 stars,`positive` = 4-5 stars. Collapsing a star-balanced sample this way yields a natural 2:1:2class distribution, which section 4 below shows.**Problem.** Supervised 3-class sentiment classification: predict whether a customer reviewis negative, neutral or positive from its raw text. The neutral class is the genuinely hardone because it overlaps both poles, and that is exactly what makes the four-model comparisonin Q2 to Q4 diverge rather than all scoring the same.**What this notebook does.** The full EDA plus all data preparation for the 5-mark EDAcomponent, and it saves the cleaned dataset that the Q2, Q3 and Q4 notebooks load. Charts arewritten to `eda_outputs/` for the report.Run the cells top to bottom.

In [1]:
from pathlib import Path

import matplotlib
matplotlib.use("Agg")               # headless backend: save figures, do not pop up windows
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import chi2

from clean_review import clean_review   # standard text-cleaning (no neg_/emoji feature-engineering)

In [2]:
# ---- paths ----
# Walk up from the current working directory to locate Part_B, so the notebook works
# whether the kernel starts in this folder, in Part_B, or at the repository root.
def find_part_b_dir():
    marker = Path("data") / "yelp_review_full_raw_30k.csv"
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).exists():
            return candidate
        if (candidate / "Part_B" / marker).exists():
            return candidate / "Part_B"
    raise FileNotFoundError(
        "Could not locate Part_B/data/yelp_review_full_raw_30k.csv from " + str(Path.cwd())
    )


PART_B_DIR = find_part_b_dir()
HERE = PART_B_DIR / "Q1_Dataset_EDA"
DATA_DIR = PART_B_DIR / "data"
RAW_FILE = DATA_DIR / "yelp_review_full_raw_30k.csv"
CLEAN_FILE = DATA_DIR / "yelp_clean.csv"
OUT_DIR = HERE / "eda_outputs"
OUT_DIR.mkdir(exist_ok=True)

TEXT_COL = "review"
LABEL_COL = "rating"      # raw star rating (1..5), kept in the saved dataset
SENT_COL = "sentiment"    # 3-class target derived from rating: negative/neutral/positive

print(f"Part_B directory : {PART_B_DIR}")
print(f"Raw data file    : {RAW_FILE}")

Part_B directory : C:\Users\yingx\Desktop\TextAssignment\Part_B
Raw data file    : C:\Users\yingx\Desktop\TextAssignment\Part_B\data\yelp_review_full_raw_30k.csv


In [3]:
def section(title):
    print(f"\n{'=' * 60}\n{title}\n{'=' * 60}")


def to_sentiment(rating):
    # Collapse the 1..5 star rating into 3 sentiment classes.
    # 1-2 -> negative, 3 -> neutral, 4-5 -> positive.
    return np.where(rating <= 2, "negative", np.where(rating == 3, "neutral", "positive"))

In [4]:
def chi2_top_words_per_class(texts, labels, top_n=12):
    """Chi-square test: words most associated with each sentiment class.

    For each class (negative/neutral/positive) we run a one-vs-rest chi-square over
    TF-IDF features (same TF-IDF config as the models, so the EDA matches the
    pipeline). We then keep only words that are OVER-represented in that class
    (mean TF-IDF inside the class > outside), so 'negative' surfaces genuinely
    angry words and 'positive' happy ones, and rank those by chi-square score.
    Returns {class: [(word, score), ...]} ordered negative -> neutral -> positive.
    """
    vec = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    X = vec.fit_transform(texts)
    features = np.array(vec.get_feature_names_out())

    result = {}
    for label in sorted(labels.unique()):     # negative < neutral < positive (alphabetical = neg->pos)
        mask = (labels == label).values
        target = mask.astype(int)
        scores, _ = chi2(X, target)

        # direction: keep only words more frequent INSIDE this class than outside
        mean_in = np.asarray(X[mask].mean(axis=0)).ravel()
        mean_out = np.asarray(X[~mask].mean(axis=0)).ravel()
        characteristic = mean_in > mean_out

        order = np.argsort(scores)[::-1]
        picks = [i for i in order if characteristic[i]][:top_n]
        result[label] = [(features[i], float(scores[i])) for i in picks]
    return result

## 1. Load the datasetRead the raw 30k sample and derive the 3-class sentiment target from the star rating.

In [5]:
df = pd.read_csv(RAW_FILE)
df[SENT_COL] = to_sentiment(df[LABEL_COL])   # 3-class target derived from stars

section("1. Dataset Overview")
print(f"Source file : {RAW_FILE.name}")
print(f"Rows        : {df.shape[0]}")
print(f"Columns     : {list(df.columns)}")
print("\nSample records:")
print(df.head(3).to_string())


1. Dataset Overview
Source file : yelp_review_full_raw_30k.csv
Rows        : 30000
Columns     : ['review', 'rating', 'sentiment']

Sample records:
                                                                                                                                                                                                                                                                                                                                                         review  rating sentiment
0                                                                                                                                                                                                 HORRIBLE PHARMACY!!! If you are in a hurry or in no mood to wait for your prescription, DO NOT COME HERE! Told me 15 mins and it's been 40. Still counting...       1  negative
1                                                                                                              

## 2. Data qualityCheck for missing values and exact duplicate review texts before any cleaning.

In [6]:
section("2. Missing Values")
print(df[[TEXT_COL, LABEL_COL]].isnull().sum())

section("3. Duplicate Reviews")
print(f"Exact duplicate review texts: {df.duplicated(subset=[TEXT_COL]).sum()}")


2. Missing Values
review    0
rating    0
dtype: int64

3. Duplicate Reviews
Exact duplicate review texts: 0


## 3. Class distributionThe raw star ratings are balanced at 6,000 each, but collapsing to three sentiment classesgives a natural 2:1:2 imbalance. This matters for Q4: it is why we report macro-averagedmetrics rather than plain accuracy, and why several models use `class_weight="balanced"`.Saves `eda_outputs/class_distribution.png`.

In [7]:
section("4. Class Distribution (3-class sentiment)")
order = ["negative", "neutral", "positive"]
dist = df[SENT_COL].value_counts().reindex(order)
print(dist)
print("(Underlying stars are balanced 6k each; the 2:1:2 split comes from the")
print(" 1-2 / 3 / 4-5 collapse -> handle with class_weight + macro metrics in Q4.)")

plt.figure(figsize=(7, 4))
plt.bar(order, dist.values, color=["#C0392B", "#C9B037", "#2E8B57"])
plt.title("Yelp review count per sentiment class (3-class)")
plt.xlabel("Sentiment class")
plt.ylabel("Number of reviews")
plt.tight_layout()
plt.savefig(OUT_DIR / "class_distribution.png", dpi=120)
plt.close()
print(f"Saved chart -> {OUT_DIR / 'class_distribution.png'}")


4. Class Distribution (3-class sentiment)
sentiment
negative    12000
neutral      6000
positive    12000
Name: count, dtype: int64
(Underlying stars are balanced 6k each; the 2:1:2 split comes from the
 1-2 / 3 / 4-5 collapse -> handle with class_weight + macro metrics in Q4.)


Saved chart -> C:\Users\yingx\Desktop\TextAssignment\Part_B\Q1_Dataset_EDA\eda_outputs\class_distribution.png


## 4. Data preparation and cleaningAll data preparation happens here, via `clean_review.py`. That cleaner lowercases, strips URLsand HTML, expands negation contractions (`won't` becomes `will not`), removes emoticons,tokenises to letters only, removes stopwords **but deliberately keeps 11 negation words**(`not`, `no`, `never`, and so on), then lemmatises using part-of-speech information.Keeping negation words matters for sentiment: "not good" must not become "good".

In [8]:
section("5. Data Preparation / Cleaning")
print("Applying clean_review (lowercase, strip URLs/HTML, expand contractions,")
print("keep negation words, remove emoticons, remove stopwords, lemmatize)...")
df = df.dropna(subset=[TEXT_COL, LABEL_COL]).copy()
df["clean_text"] = df[TEXT_COL].apply(clean_review)
before = len(df)
df = df[df["clean_text"].str.strip() != ""].copy()
print(f"Rows before cleaning: {before}")
print(f"Rows after  cleaning: {len(df)}  (empty rows dropped: {before - len(df)})")


5. Data Preparation / Cleaning
Applying clean_review (lowercase, strip URLs/HTML, expand contractions,
keep negation words, remove emoticons, remove stopwords, lemmatize)...


Rows before cleaning: 30000
Rows after  cleaning: 29998  (empty rows dropped: 2)


## 5. Text length analysis, before against after cleaningShows how much the cleaning step compresses each review, and whether review length differs bysentiment class.Saves `eda_outputs/length_distribution.png`.

In [9]:
df["char_count"] = df[TEXT_COL].astype(str).str.len()
df["words_before"] = df[TEXT_COL].astype(str).str.split().str.len()
df["words_after"] = df["clean_text"].str.split().str.len()

section("6. Text Length Summary")
print(df[["char_count", "words_before", "words_after"]].describe().round(1))

section("7. Average Cleaned Word Count by Sentiment Class")
print(df.groupby(SENT_COL)["words_after"].mean().round(1).reindex(["negative", "neutral", "positive"]))

plt.figure(figsize=(8, 4))
plt.hist(df["words_before"], bins=50, color="#C44E52", alpha=0.7, label="before cleaning")
plt.hist(df["words_after"], bins=50, color="#4C72B0", alpha=0.7, label="after cleaning")
plt.title("Review length distribution (words)")
plt.xlabel("Words per review")
plt.ylabel("Number of reviews")
plt.xlim(0, 400)
plt.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "length_distribution.png", dpi=120)
plt.close()
print(f"Saved chart -> {OUT_DIR / 'length_distribution.png'}")


6. Text Length Summary
       char_count  words_before  words_after
count     29998.0       29998.0      29998.0
mean        735.9         134.7         69.0
std         661.8         121.0         61.1
min           2.0           1.0          1.0
25%         289.0          53.0         28.0
50%         541.0         100.0         51.0
75%         964.0         177.0         90.0
max        5057.0         991.0        521.0

7. Average Cleaned Word Count by Sentiment Class
sentiment
negative    77.1
neutral     72.1
positive    59.4
Name: words_after, dtype: float64


Saved chart -> C:\Users\yingx\Desktop\TextAssignment\Part_B\Q1_Dataset_EDA\eda_outputs\length_distribution.png


## 6. Chi-square: the words that actually separate the classesThis replaces a frequency-based top-words chart and word clouds, which were dominated byshared nouns that appear in every class regardless of sentiment. A chi-square test insteadsurfaces the words that genuinely *discriminate* between classes, filtered to keep only wordsover-represented inside each class.It also gives visual evidence for the central claim of this project: the neutral classoverlaps both poles, so the task is not trivially separable.Saves `eda_outputs/chi2_top_words_per_class.png`.

In [10]:
section("8. Chi-square: most distinctive words per sentiment class (negative .. positive)")
chi_words = chi2_top_words_per_class(df["clean_text"], df[SENT_COL])
for label in sorted(chi_words):
    words = ", ".join(w for w, _ in chi_words[label])
    print(f"{label}: {words}")


8. Chi-square: most distinctive words per sentiment class (negative .. positive)


negative: horrible, rude, bad, tell, terrible, no, manager, not even, customer, poor, ask, minute
neutral: decent, three star, not bad, ok, pretty, pretty good, okay, average, good, bit, hit miss, though
positive: great, love, delicious, amaze, awesome, favorite, highly recommend, excellent, best, perfect, amazing, friendly


In [11]:
# negative (red) -> neutral (amber) -> positive (green) spectrum, one per panel
spectrum = ["#C0392B", "#C9B037", "#2E8B57"]
n_classes = len(chi_words)
fig, axes = plt.subplots(n_classes, 1, figsize=(9, 2.6 * n_classes))
for ax, label, color in zip(axes, sorted(chi_words), spectrum):
    pairs = chi_words[label][::-1]           # smallest at bottom, largest on top
    ax.barh([w for w, _ in pairs], [s for _, s in pairs], color=color)
    ax.set_title(f"{label} - most distinctive words (chi-square)", fontsize=10)
    ax.tick_params(axis="y", labelsize=8)
fig.supxlabel("chi-square score (word vs this sentiment class)")
fig.tight_layout()
fig.savefig(OUT_DIR / "chi2_top_words_per_class.png", dpi=120)
plt.close(fig)
print("Saved chi-square chart -> chi2_top_words_per_class.png")

Saved chi-square chart -> chi2_top_words_per_class.png


## 7. Save the cleaned datasetThis is the handover point to the rest of Part B. Every Q2, Q3 and Q4 notebook loads`data/yelp_clean.csv` and uses the `clean_text` and `sentiment` columns.The raw `rating` column is kept as well, so a binary or 5-class variant could still be derivedlater without re-running the cleaning.

In [12]:
out = df[[TEXT_COL, "clean_text", LABEL_COL, SENT_COL]].copy()
out.to_csv(CLEAN_FILE, index=False)
section("9. Saved Clean Dataset")
print(f"Saved -> {CLEAN_FILE}")
print(f"Columns: {list(out.columns)}  (label = '{SENT_COL}': negative/neutral/positive; '{LABEL_COL}' kept raw)")
print(f"Rows   : {len(out)}")


9. Saved Clean Dataset
Saved -> C:\Users\yingx\Desktop\TextAssignment\Part_B\data\yelp_clean.csv
Columns: ['review', 'clean_text', 'rating', 'sentiment']  (label = 'sentiment': negative/neutral/positive; 'rating' kept raw)
Rows   : 29998


## 8. The four proposed modelsOne per group member, chosen to span genuinely different learning strategies rather than fourvariations on the same idea: two linear models, one probabilistic model, and one non-linearensemble. Q2 builds them, Q3 tunes them, Q4 compares them.

In [13]:
section("10. Proposed Predictive Models (4 members)")
proposals = [
    "Logistic Regression - strong, fast linear baseline for sparse TF-IDF text; "
    "weighs all word features together, well suited to high-dimensional text.",
    "Linear SVM (LinearSVC) - maximum-margin linear classifier, very effective on "
    "high-dimensional sparse text features.",
    "Random Forest - non-linear ensemble; included to contrast tree-based learning "
    "against the linear models on sparse text.",
    "Multinomial Naive Bayes - probabilistic word-count model; classic, fast text "
    "classification baseline.",
]
for i, p in enumerate(proposals, 1):
    print(f"  {i}. {p}")

print(f"\nAll EDA charts saved in: {OUT_DIR}")


10. Proposed Predictive Models (4 members)
  1. Logistic Regression - strong, fast linear baseline for sparse TF-IDF text; weighs all word features together, well suited to high-dimensional text.
  2. Linear SVM (LinearSVC) - maximum-margin linear classifier, very effective on high-dimensional sparse text features.
  3. Random Forest - non-linear ensemble; included to contrast tree-based learning against the linear models on sparse text.
  4. Multinomial Naive Bayes - probabilistic word-count model; classic, fast text classification baseline.

All EDA charts saved in: C:\Users\yingx\Desktop\TextAssignment\Part_B\Q1_Dataset_EDA\eda_outputs
